
# 練習問題: 行列ベクトル積を for/do で並列化する

## 目標

work-sharing 構文 (`for` / `do`) を使って, 密な行列ベクトル積 `y = A x` を並列化する. 行ごとの計算を `#pragma omp parallel for` (Fortran は `!$omp parallel do`) でスレッドに分担させる. 各行の内積は局所変数に貯めるので `reduction` は不要であることを理解する.

## 背景

`n × n` の行列 `A` と長さ `n` のベクトル `x` の積 `y = A x` は

```
y[i] = Σ_j A[i][j] * x[j]   (i = 0, ..., n-1)
```

で定義される. 行列 `A` は1次元配列に `A[i*n + j]` として格納している. 各行 `i` の計算は他の行と独立なので, 外側の `i` ループを並列化できる.

## 課題

`matvec.cpp` (または `matvec.f90`) は, サイズ `n` (コマンドライン引数, 既定 4000) で `A[i*n+j] = 1`, `x[j] = 1` と初期化する. このとき正解は `y[i] = n` (全要素 `n`) になる.

コメント `TODO` の指示に従って **OpenMP の指示を追加** し, 行ループを並列化せよ.

- C++: 外側の `i` ループの直前に `#pragma omp parallel for` を1行加える. 内積の和 `s` はループ内で宣言してあるので, 反復ごと (= スレッドごと) に別々の変数になる.
- Fortran: 外側の `do i` ループを `!$omp parallel do private(j, s)` と `!$omp end parallel do` で囲む. 内側ループ変数 `j` と和 `s` をスレッドごとに別々にするため `private` 節に並べる.

並列化しても結果 (`y[i] = n`, 検算 `OK`) は変わらない. それ以外のコードを変更する必要はない.

**発展 (任意):** 初期化の二重ループ (`A[i*n+j] = 1`) は `collapse(2)` を付けて2重ループをまとめて並列化できる. 余裕があれば試してみよ. ただし採点の対象 (穴あき) は行列ベクトル積の `parallel for` の方である.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore matvec.cpp -o matvec.exe

# Fortran
nvfortran -fast -mp=multicore matvec.f90 -o matvec.exe
```

```
OMP_NUM_THREADS=4 ./matvec.exe
OMP_NUM_THREADS=8 ./matvec.exe 8000
```

## 期待される結果

```
n = 4000, y[0] = 4000.000000 (expected 4000): OK
```

`OK` が表示されれば正しく計算できている. スレッド数 (`OMP_NUM_THREADS`) を変えても結果は変わらず, `n` を大きくすると計算が重くなるので並列化の効果 (実行時間の短縮) を `time` などで確認できる.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ matvec.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char **argv) {
  // 行列・ベクトルのサイズ (コマンドライン引数, 既定 4000)
  int n = (argc > 1) ? atoi(argv[1]) : 4000;

  double *A = (double *)malloc((size_t)n * n * sizeof(double));
  double *x = (double *)malloc((size_t)n * sizeof(double));
  double *y = (double *)malloc((size_t)n * sizeof(double));

  // 検算しやすい初期化: A[i*n+j] = 1, x[j] = 1 とすると y[i] = n になる.
  // (この初期化ループは collapse(2) で並列化してもよい)
  for (int i = 0; i < n; i++) {
    for (int j = 0; j < n; j++) {
      A[(size_t)i * n + j] = 1.0;
    }
  }
  for (int j = 0; j < n; j++) {
    x[j] = 1.0;
  }

  // 行列ベクトル積 y = A x
  // TODO: 下の行 (外側の i ループ) の直前に #pragma omp parallel for を1行追加し, 行ごとの計算をスレッドで分担せよ.
  for (int i = 0; i < n; i++) {
    double s = 0.0;  // 行ごとの局所アキュムレータ (reduction 不要)
    for (int j = 0; j < n; j++) {
      s += A[(size_t)i * n + j] * x[j];
    }
    y[i] = s;
  }

  // 検算: すべての y[i] が n に等しいはず
  int ok = 1;
  for (int i = 0; i < n; i++) {
    if (y[i] != (double)n) {
      ok = 0;
      break;
    }
  }
  printf("n = %d, y[0] = %f (expected %d): %s\n",
         n, y[0], n, ok ? "OK" : "NG");

  free(A);
  free(x);
  free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore matvec.cpp -o matvec_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matvec_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ matvec.f90
program matvec
  implicit none
  integer :: n, i, j
  real(8), allocatable :: A(:), x(:), y(:)
  real(8) :: s
  logical :: ok
  character(len=32) :: arg

  ! 行列・ベクトルのサイズ (コマンドライン引数, 既定 4000)
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg)
     read(arg, *) n
  else
     n = 4000
  end if

  allocate(A(0:n*n-1), x(0:n-1), y(0:n-1))

  ! 検算しやすい初期化: A(i*n+j) = 1, x(j) = 1 とすると y(i) = n になる.
  ! (この初期化ループは collapse(2) で並列化してもよい)
  do i = 0, n - 1
     do j = 0, n - 1
        A(i*n + j) = 1.0d0
     end do
  end do
  do j = 0, n - 1
     x(j) = 1.0d0
  end do

  ! 行列ベクトル積 y = A x
  ! TODO: 下の外側の do ループを !$omp parallel do private(j, s) ... !$omp end parallel do で囲み, 行ごとの計算をスレッドで分担せよ.
  do i = 0, n - 1
     s = 0.0d0  ! 行ごとの局所アキュムレータ (reduction 不要)
     do j = 0, n - 1
        s = s + A(i*n + j) * x(j)
     end do
     y(i) = s
  end do
  ! TODO: 上で始めた parallel do を閉じる (!$omp end parallel do).

  ! 検算: すべての y(i) が n に等しいはず
  ok = .true.
  do i = 0, n - 1
     if (y(i) /= real(n, 8)) then
        ok = .false.
        exit
     end if
  end do
  if (ok) then
     print "(a,i0,a,f0.6,a,i0,a)", "n = ", n, ", y(0) = ", y(0), &
          " (expected ", n, "): OK"
  else
     print "(a,i0,a,f0.6,a,i0,a)", "n = ", n, ", y(0) = ", y(0), &
          " (expected ", n, "): NG"
  end if

  deallocate(A, x, y)
end program matvec

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore matvec.f90 -o matvec_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matvec_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:matvec.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:matvec.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:matvec.cpp}